In [1]:
import os

try:
    import google.colab
    print("Running in Google Colab")
    from google.colab import drive
    drive.mount('/content/drive')
    os.chdir('/content/drive/MyDrive/dugout-prophet')
    os.system('pip install pybaseball')
except ImportError:
    print("Not in Google Colab")

Mounted at /content/drive


In [16]:
# Check for local CSV; if missing, download from Firebase using pyasebase/pyrebase
import os
import pandas as pd
from pybaseball import batting_stats
import numpy as np

csv_path = "./data/batting_stats.csv"

if os.path.exists(csv_path):
    print(f"Found {csv_path}, reading into DataFrame.")
    data = pd.read_csv(csv_path)

else:
    print(f"{csv_path} not found — attempting to download using pybaseball.")
    qual = 50
    start_season = 2015
    end_season = 2025
    data = batting_stats(start_season=start_season, end_season=end_season, qual=qual)
    data.to_csv('./data/batting_stats.csv', index=False)


Found ./data/batting_stats.csv, reading into DataFrame.


In [17]:
score_map = {
    'R': 0.75,
    '1B': 1,
    '2B': 1.5,
    '3B': 2,
    'HR': 3,
    'RBI': 0.75,
    'BB':1, 
    'SO':-0.5,
    'HBP': 1,
    'SB': 1,
    'CS': -2,
    'GDP': -2,
}

In [18]:
# Process data for 3D tensor - organized by player-season combination
import torch
import numpy as np
import pandas as pd

# Select relevant columns
features = list(score_map.keys())

# Qualification parameter: minimum IP in target season to include in training set
min_qual_pa = 50  # Set to > 0 to filter by innings pitched (e.g., 50 for minimum 50 IP)

# Ensure data has the necessary columns
required_cols = features + ['IDfg', 'Season']
data_processed = data.reindex(columns=[c for c in data.columns if c in required_cols]).fillna(0)


In [19]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# Multivariate Transformer Model with Multihead Attention for next year feature prediction
class MultiheadTransformerPredictor(nn.Module):
    def __init__(self, input_dim, num_features, d_model=64, nhead=4, num_layers=2, 
                 dim_feedforward=256, dropout=0.1, nlookbacks=5):
        """
        Multivariate Transformer model for predicting next year's features.
        
        Args:
            input_dim: Number of input features (equal to num_features)
            num_features: Number of features to predict for next year
            d_model: Dimension of the model (embedding dimension)
            nhead: Number of attention heads
            num_layers: Number of transformer layers
            dim_feedforward: Dimension of feedforward networks
            dropout: Dropout rate
            nlookbacks: Number of lookback years/seasons to use for prediction
        """
        super(MultiheadTransformerPredictor, self).__init__()
        
        self.input_dim = input_dim
        self.num_features = num_features
        self.d_model = d_model
        self.nlookbacks = nlookbacks
        
        # Input projection layer - transforms raw features to d_model dimension
        self.input_projection = nn.Linear(input_dim, d_model)
        
        # Positional encoding for sequence
        self.register_buffer('pos_encoding', self._generate_pos_encoding(d_model, nlookbacks))
        
        # Transformer encoder with multihead attention
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        # Output layers for feature prediction
        self.fc_out = nn.Sequential(
            nn.Linear(d_model, dim_feedforward),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(dim_feedforward, num_features)
        )
    
    def _generate_pos_encoding(self, d_model, seq_len):
        """Generate positional encoding for transformer."""
        position = torch.arange(seq_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * 
                            (-torch.log(torch.tensor(10000.0)) / d_model))
        pe = torch.zeros(seq_len, d_model)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        return pe.unsqueeze(0)  # Shape: (1, seq_len, d_model)
    
    def forward(self, x):
        """
        Forward pass for the transformer model.
        
        Args:
            x: Input tensor of shape (batch_size, nlookbacks, input_dim)
               Where nlookbacks is the number of historical seasons/years
               
        Returns:
            output: Predicted features for next year of shape (batch_size, num_features)
        """
        batch_size, seq_len, _ = x.shape
        
        # Project input features to d_model dimension
        x = self.input_projection(x)  # (batch_size, seq_len, d_model)
        
        # Add positional encoding
        x = x + self.pos_encoding[:, :seq_len, :]  # (batch_size, seq_len, d_model)
        
        # Apply transformer encoder with multihead attention
        transformer_out = self.transformer_encoder(x)  # (batch_size, seq_len, d_model)
        
        # Use the last sequence element (most recent year) for prediction
        # Alternatively, could use mean pooling: transformer_out.mean(dim=1)
        last_output = transformer_out[:, -1, :]  # (batch_size, d_model)
        
        # Predict next year's features
        output = self.fc_out(last_output)  # (batch_size, num_features)
        
        return output


# Initialize the multihead transformer model
num_features = len(features)
nlookbacks = 5  # Number of seasons to look back for prediction

model = MultiheadTransformerPredictor(
    input_dim=num_features,
    num_features=num_features,
    d_model=64,
    nhead=4,  # 4 attention heads
    num_layers=2,  # 2 transformer layers
    dim_feedforward=256,
    dropout=0.1,
    nlookbacks=nlookbacks
)

print(f"Model created with {num_features} features and {nlookbacks} lookback seasons")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")

Model created with 12 features and 5 lookback seasons
Total parameters: 120,524


In [20]:
from torch.utils.data import Dataset, DataLoader

class MultivariateBattingDataset(Dataset):
    """
    Creates training sequences for multivariate transformer model.
    For each player, generates sliding windows of nlookbacks seasons
    with corresponding next-year target features.
    """
    def __init__(self, data, features, player_col='IDfg', season_col='Season', 
                 nlookbacks=5, min_qual_pa=50):
        """
        Args:
            data: DataFrame with batting statistics
            features: List of feature column names
            player_col: Column name for player ID
            season_col: Column name for season year
            nlookbacks: Number of lookback years for sequences
            min_qual_pa: Minimum plate appearances to include a player-season
        """
        self.data = data.copy()
        self.features = features
        self.player_col = player_col
        self.season_col = season_col
        self.nlookbacks = nlookbacks
        self.min_qual_pa = min_qual_pa
        
        # Filter by minimum qualification
        if 'PA' in self.data.columns:
            self.data = self.data[self.data['PA'] >= min_qual_pa]
        
        self.sequences = self._create_sequences()
    
    def _create_sequences(self):
        """Create training sequences from player-season data."""
        sequences = []
        
        # Group by player
        for player_id, player_data in self.data.groupby(self.player_col):
            # Sort by season
            player_data = player_data.sort_values(self.season_col).reset_index(drop=True)
            
            # Extract feature values
            feature_values = player_data[self.features].values  # (num_seasons, num_features)
            num_seasons = len(feature_values)
            
            # Create sliding windows with nlookbacks + 1 (for target)
            for i in range(num_seasons - self.nlookbacks):
                # Input: nlookbacks seasons
                input_seq = feature_values[i:i + self.nlookbacks]  # (nlookbacks, num_features)
                
                # Target: next year's features
                target_seq = feature_values[i + self.nlookbacks]   # (num_features,)
                
                sequences.append({
                    'input': torch.tensor(input_seq, dtype=torch.float32),
                    'target': torch.tensor(target_seq, dtype=torch.float32),
                    'player_id': player_id,
                    'season': player_data.iloc[i + self.nlookbacks][self.season_col]
                })
        
        return sequences
    
    def __len__(self):
        return len(self.sequences)
    
    def __getitem__(self, idx):
        seq = self.sequences[idx]
        return seq['input'], seq['target']


# Create training dataset
print("Creating multivariate training dataset...")
dataset = MultivariateBattingDataset(
    data=data_processed,
    features=features,
    player_col='IDfg',
    season_col='Season',
    nlookbacks=nlookbacks,
    min_qual_pa=min_qual_pa
)

print(f"Dataset created with {len(dataset)} sequences")
print(f"Each sequence: {nlookbacks} lookback years × {len(features)} features")
print(f"Target: {len(features)} features for next year")

# Create data loaders for training and validation
batch_size = 32
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"\nDataLoaders created:")
print(f"  Training set: {len(train_dataset)} sequences ({train_size/len(dataset)*100:.1f}%)")
print(f"  Test set: {len(test_dataset)} sequences ({test_size/len(dataset)*100:.1f}%)")
print(f"  Batch size: {batch_size}")

Creating multivariate training dataset...
Dataset created with 1110 sequences
Each sequence: 5 lookback years × 12 features
Target: 12 features for next year

DataLoaders created:
  Training set: 888 sequences (80.0%)
  Test set: 222 sequences (20.0%)
  Batch size: 32


In [21]:
dataset.sequences

[{'input': tensor([[ 82.,  79.,  43.,   1.,  35.,  89.,  44., 137.,   7.,  13.,   8.,  19.],
          [ 89.,  72.,  21.,   0.,  40.,  98.,  64., 163.,   4.,  15.,   5.,  11.],
          [ 74.,  54.,  19.,   1.,  27.,  76.,  83., 125.,  14.,   4.,   3.,  10.],
          [ 54.,  51.,  18.,   0.,  18.,  59.,  48., 112.,   8.,   9.,   4.,  10.],
          [ 63.,  70.,  19.,   2.,  21.,  67.,  40., 106.,  12.,   1.,   2.,   9.]]),
  'target': tensor([16., 23.,  9.,  1.,  4., 12., 11., 42.,  4.,  1.,  1.,  1.]),
  'player_id': 785,
  'season': np.int64(2020)},
 {'input': tensor([[ 85.,  85.,  22.,   0.,  40.,  95.,  50.,  72.,   6.,   5.,   3.,  15.],
          [ 71., 109.,  19.,   0.,  31., 119.,  49.,  75.,   2.,   4.,   0.,  24.],
          [ 53., 103.,  17.,   0.,  23., 101.,  37.,  93.,   2.,   3.,   0.,  26.],
          [ 50.,  75.,  20.,   0.,  19.,  64.,  28.,  65.,   2.,   1.,   0.,  12.],
          [ 55.,  75.,  22.,   0.,  23.,  93.,  43.,  68.,   3.,   3.,   0.,  21.]]),
  'targ